In [1]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
ROOT_DIR = Path("/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome")
INPUT_DIR = ROOT_DIR / "input_data/HBN"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
output_dir = ROOT_DIR / "output_data" / "cleaned" / "HBN"
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
general_ses = pd.read_csv(INPUT_DIR / "HBN_General_SES.csv")
print(general_ses["GUID"].nunique())
general_ses.head(2)

6220


,GUID,General_SES
0,NDARAA066GKK,3.18
1,NDARAA075AMK,1.10


In [4]:
# folder containing noddi data from cubic
# generated with: bash \
# 	unzip_files.sh \
# 	/cbica/projects/pennlinc_rbc/datasets/LINC_HBN/derivatives/QSIRECON-1-1-0_BUNDLE-STATS_zipped \
# 	/cbica/projects/macedo_long_wm/data/noddi \
# 	"qsirecon/derivatives/qsirecon-wmNODDI/sub-*/ses-1/dwi/sub-*_bundles-DSIStudio_scalarstats*"

noddi_dir = INPUT_DIR / "noddi"

tsv_files = sorted(noddi_dir.glob("sub-*_ses-1_acq-64dir_space-ACPC_bundles-DSIStudio_scalarstats.tsv"))
allowed_ids = set("sub-" + general_ses["GUID"].astype(str)) 

dfs = []
keep_metrics = {"icvf", "isovf", "od"}

for f in tsv_files:
    df = pd.read_csv(f, sep="\t")
    df_filt = df[df["variable_name"].isin(keep_metrics)].copy()
    df_filt["metric"] = "NODDI_" + df_filt["variable_name"].astype(str)
    df_filt = df_filt[["bundle", "metric", "mean", "subject_id"]] 
    df_filt = df_filt[df_filt["subject_id"].isin(allowed_ids)]
    dfs.append(df_filt)

# Concatenate all subjects into one DataFrame
all_noddi = pd.concat(dfs, ignore_index=True)

# save to cleaned/HBN

output_path = output_dir / "noddi_data.csv"
all_noddi.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {all_noddi.shape}")
all_noddi.head()

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/HBN/noddi_data.csv | shape: (327549, 4)


,bundle,metric,mean,subject_id
0,Association_InferiorLongitudinalFasciculusR,NODDI_icvf,0.544184,sub-NDARAA396TWZ
1,Association_InferiorLongitudinalFasciculusR,NODDI_isovf,0.177847,sub-NDARAA396TWZ
2,Association_InferiorLongitudinalFasciculusR,NODDI_od,0.413321,sub-NDARAA396TWZ
3,Association_ArcuateFasciculusR,NODDI_icvf,0.542660,sub-NDARAA396TWZ
4,Association_ArcuateFasciculusR,NODDI_isovf,0.161884,sub-NDARAA396TWZ


In [5]:
all_noddi["subject_id"].nunique()

1698

In [6]:
# folder containing macrostructure data from cubic
# generated with: bash \
# 	unzip_files.sh \
# 	/cbica/projects/pennlinc_rbc/datasets/LINC_HBN/derivatives/QSIRECON-1-1-0_BUNDLE-STATS_zipped \
# 	/cbica/projects/macedo_long_wm/data/macro \
# 	"qsirecon/derivatives/qsirecon-MSMTAutoTrack/sub-*/ses-1/dwi/sub-*msmt_bundlestats*"

macro_dir = INPUT_DIR / "macro"

csv_files = sorted(macro_dir.glob("*.csv"))
cols_keep = ['bundle_name', 'mean_length_mm', 'span_mm', 'curl', 'elongation', 'total_volume_mm3', '1st_quarter_volume_mm3',
             '2nd_and_3rd_quarter_volume_mm3', '4th_quarter_volume_mm3','total_surface_area_mm2', 'total_radius_of_end_regions_mm',
             'total_area_of_end_regions_mm2', 'irregularity', 'area_of_end_region_1_mm2', 'radius_of_end_region_1_mm',
             'volume_of_end_branches_1', 'area_of_end_region_2_mm2', 'radius_of_end_region_2_mm', 'volume_of_end_branches_2',
             'subject_id']
dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df = df[cols_keep].copy()
    df = df[df["subject_id"].isin(allowed_ids)]
    dfs.append(df)

macro_wide = pd.concat(dfs, ignore_index=True)

# long format: bundle_name + metric + mean + subject_id 
value_cols = [c for c in cols_keep if c not in ["bundle_name", "subject_id"]]

macro_long = macro_wide.melt(id_vars=["subject_id", "bundle_name"], value_vars=value_cols,
                             var_name="metric", value_name="mean").copy()

macro_long.rename(columns={"bundle_name": "bundle"}, inplace=True)

output_path = output_dir / "macro_data.csv"
macro_long.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {macro_long.shape}")
macro_long.head()

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/HBN/macro_data.csv | shape: (4130280, 4)


,subject_id,bundle,metric,mean
0,sub-NDARAA306NT2,Association_CingulumL,mean_length_mm,73.0191
1,sub-NDARAA306NT2,Association_FrontalAslantTractR,mean_length_mm,63.4005
2,sub-NDARAA306NT2,Association_ArcuateFasciculusR,mean_length_mm,102.5180
3,sub-NDARAA306NT2,Association_MiddleLongitudinalFasciculusL,mean_length_mm,81.6783
4,sub-NDARAA306NT2,Association_VerticalOccipitalFasciculusR,mean_length_mm,39.9459


In [7]:
macro_long["subject_id"].nunique()

2980

In [8]:
# folder containing bundle qc data
# generated with: bash \
# 	unzip_files.sh \
# 	/cbica/projects/pennlinc_rbc/datasets/LINC_HBN/derivatives/QSIPREP-1-0-1_zipped \
# 	/cbica/projects/macedo_long_wm/data/qc \
# 	"qsiprep/sub-*/ses-1/dwi/sub-*desc-image_qc*"

qc_dir = INPUT_DIR / "qc"

qc_files = sorted(qc_dir.glob("sub-*desc-image_qc.tsv"))

qc_rows = []
for f in qc_files:
    m = re.search(r"(sub-[A-Za-z0-9]+)", f.name)
    subject_id = m.group(1)

    df_qc = pd.read_csv(f, sep="\t")
    val = df_qc["t1_neighbor_corr"].iloc[0]
    qc_rows.append({"subject_id": subject_id, "t1_neighbor_corr": val})

qc_df = pd.DataFrame(qc_rows)
qc_df["t1_neighbor_corr"] = pd.to_numeric(qc_df["t1_neighbor_corr"], errors="coerce")

output_path = output_dir / "qc_data.csv"
qc_df.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {qc_df.shape}")
qc_df.head()

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/HBN/qc_data.csv | shape: (3298, 2)


,subject_id,t1_neighbor_corr
0,sub-NDARAA306NT2,0.975595
1,sub-NDARAA396TWZ,0.975088
2,sub-NDARAA504CRN,0.982532
3,sub-NDARAA536PTU,0.982706
4,sub-NDARAA947ZG5,0.971159


In [9]:
qc_df["subject_id"].nunique()

3298

In [10]:
# combine macro + NODDI into one long dataframe
all_features_long = pd.concat([all_noddi, macro_long], ignore_index=True)

output_path = output_dir / "all_macro_noddi_long.csv"
all_features_long.to_csv(output_path, index=False)

print(f"[SAVE] → {output_path} | shape: {all_features_long.shape}")
all_features_long.head()

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/cleaned/HBN/all_macro_noddi_long.csv | shape: (4457829, 4)


,bundle,metric,mean,subject_id
0,Association_InferiorLongitudinalFasciculusR,NODDI_icvf,0.544184,sub-NDARAA396TWZ
1,Association_InferiorLongitudinalFasciculusR,NODDI_isovf,0.177847,sub-NDARAA396TWZ
2,Association_InferiorLongitudinalFasciculusR,NODDI_od,0.413321,sub-NDARAA396TWZ
3,Association_ArcuateFasciculusR,NODDI_icvf,0.542660,sub-NDARAA396TWZ
4,Association_ArcuateFasciculusR,NODDI_isovf,0.161884,sub-NDARAA396TWZ


In [11]:
all_features_long["subject_id"].nunique()

2980

In [12]:
participants = pd.read_csv(INPUT_DIR / "participants.tsv", sep="\t")
participants = participants[["participant_id", "Sex", "Age", "site"]]
participants.head(3)

,participant_id,Sex,Age,site
0,sub-NDARAA075AMK,1.0,6.728040,si
1,sub-NDARAA112DMH,0.0,5.545744,ru
2,sub-NDARAA117NEJ,0.0,7.475929,ru


In [13]:
participants["participant_id"].nunique()

3887

In [14]:
df_long = all_features_long.copy()

# create a combined feature name: FEATURES_bundle_metric
df_long["feature_name"] = ("FEATURES_" + df_long["bundle"].astype(str) + "_" + df_long["metric"].astype(str))

# pivot to wide: one row per subject_id, one column per feature_name
df_wide = df_long.pivot(index="subject_id", columns="feature_name", values="mean")
df_wide = df_wide.reset_index()

# make subject id col consistent across dfs
general_ses = general_ses.copy()
general_ses["subject_id"] = "sub-" + general_ses["GUID"].astype(str)

# only keep subjects present in the features df
df_merged = df_wide.merge(general_ses[["subject_id", "GUID", "General_SES"]], on="subject_id", how="inner")

# merge in participant info
participants_sub = participants[["participant_id", "Sex", "Age", "site"]].copy()
df_with_demo = df_merged.merge(participants_sub, left_on="subject_id", right_on="participant_id", how="inner")

# reorder columns so participant_id, Sex, Age, site come first
feature_cols = [c for c in df_with_demo.columns if c.startswith("FEATURES_")]
first_cols = ["participant_id", "Sex", "Age", "site", "subject_id", "GUID", "General_SES"]
other_cols = [c for c in df_with_demo.columns if c not in first_cols + feature_cols]

df_final = df_with_demo[first_cols + other_cols + feature_cols].copy()
df_final = df_final.merge(qc_df, on="subject_id", how="left")

df_final.head()

,participant_id,Sex,Age,site,subject_id,GUID,General_SES,FEATURES_Association_ArcuateFasciculusL_1st_quarter_volume_mm3,FEATURES_Association_ArcuateFasciculusL_2nd_and_3rd_quarter_volume_mm3,FEATURES_Association_ArcuateFasciculusL_4th_quarter_volume_mm3,...,FEATURES_ProjectionBrainstem_ReticularTractR_radius_of_end_region_1_mm,FEATURES_ProjectionBrainstem_ReticularTractR_radius_of_end_region_2_mm,FEATURES_ProjectionBrainstem_ReticularTractR_span_mm,FEATURES_ProjectionBrainstem_ReticularTractR_total_area_of_end_regions_mm2,FEATURES_ProjectionBrainstem_ReticularTractR_total_radius_of_end_regions_mm,FEATURES_ProjectionBrainstem_ReticularTractR_total_surface_area_mm2,FEATURES_ProjectionBrainstem_ReticularTractR_total_volume_mm3,FEATURES_ProjectionBrainstem_ReticularTractR_volume_of_end_branches_1,FEATURES_ProjectionBrainstem_ReticularTractR_volume_of_end_branches_2,t1_neighbor_corr
0,sub-NDARAA306NT2,1.0,21.216746,ru,sub-NDARAA306NT2,NDARAA306NT2,-0.32,7291.46,8462.96,7291.46,...,5.86138,15.1097,44.0507,3588.30,20.9711,33823.2,14586.60,3243.320,3243.320,0.975595
1,sub-NDARAA396TWZ,0.0,7.059890,cbic,sub-NDARAA396TWZ,NDARAA396TWZ,0.81,6763.66,16611.00,6763.66,...,7.24729,19.5331,48.3610,618.03,26.7804,12504.8,4035.01,733.374,733.374,0.975088
2,sub-NDARAA504CRN,1.0,9.165297,cbic,sub-NDARAA504CRN,NDARAA504CRN,-0.12,5734.31,11896.50,5734.31,...,6.24607,26.5954,47.4594,3951.99,32.8414,39265.6,18994.80,3580.850,3580.850,0.982532
3,sub-NDARAA536PTU,0.0,11.998402,si,sub-NDARAA536PTU,NDARAA536PTU,-0.30,7427.78,8904.00,7427.78,...,7.95888,25.0239,51.0082,5500.71,32.9828,52132.4,25593.00,3266.650,3266.650,0.982706
4,sub-NDARAA947ZG5,0.0,13.627880,cbic,sub-NDARAA947ZG5,NDARAA947ZG5,1.15,5386.58,18399.20,5386.58,...,7.51295,21.1176,48.0489,4412.88,28.6306,46302.0,22772.50,3203.230,3203.230,0.971159


In [15]:
df_final["participant_id"].nunique()

2980

In [16]:
bundles_to_keep = [
    "Association_ArcuateFasciculusL",
    "Association_ArcuateFasciculusR",
    "Association_CingulumL",
    "Association_CingulumR",
    "Association_ExtremeCapsuleL",
    "Association_ExtremeCapsuleR",
    "Association_FrontalAslantTractL",
    "Association_FrontalAslantTractR",
    "Association_HippocampusAlveusL",
    "Association_HippocampusAlveusR",
    "Association_InferiorFrontoOccipitalFasciculusL",
    "Association_InferiorFrontoOccipitalFasciculusR",
    "Association_InferiorLongitudinalFasciculusL",
    "Association_InferiorLongitudinalFasciculusR",
    "Association_MiddleLongitudinalFasciculusL",
    "Association_MiddleLongitudinalFasciculusR",
    "Association_ParietalAslantTractL",
    "Association_ParietalAslantTractR",
    "Association_SuperiorLongitudinalFasciculusL",
    "Association_SuperiorLongitudinalFasciculusR",
    "Association_UncinateFasciculusL",
    "Association_UncinateFasciculusR",
    "Association_VerticalOccipitalFasciculusL",
    "Association_VerticalOccipitalFasciculusR",
    "Cerebellum_CerebellumL",
    "Cerebellum_CerebellumR",
    "Cerebellum_InferiorCerebellarPeduncleL",
    "Cerebellum_InferiorCerebellarPeduncleR",
    "Cerebellum_MiddleCerebellarPeduncle",
    "Cerebellum_SuperiorCerebellarPeduncle",
    "Cerebellum_Vermis",
    "CommissureCorpusCallosum",
    "Projection_BasalGangliaAcousticRadiationL",
    "Projection_BasalGangliaAcousticRadiationR",
    "Projection_BasalGangliaAnsaLenticularisL",
    "Projection_BasalGangliaAnsaLenticularisR",
    "Projection_BasalGangliaAnsaSubthalamicaL",
    "Projection_BasalGangliaAnsaSubthalamicaR",
    "Projection_BasalGangliaCorticostriatalTractL",
    "Projection_BasalGangliaCorticostriatalTractR",
    "Projection_BasalGangliaFasciculusLenticularisL",
    "Projection_BasalGangliaFasciculusLenticularisR",
    "Projection_BasalGangliaFasciculusSubthalamicusL",
    "Projection_BasalGangliaFasciculusSubthalamicusR",
    "Projection_BasalGangliaFornixL",
    "Projection_BasalGangliaFornixR",
    "Projection_BasalGangliaOpticRadiationL",
    "Projection_BasalGangliaOpticRadiationR",
    "Projection_BasalGangliaThalamicRadiationL",
    "Projection_BasalGangliaThalamicRadiationR",
    "Projection_BrainstemCorticopontineTractL",
    "Projection_BrainstemCorticopontineTractR",
    "Projection_BrainstemCorticospinalTractL",
    "Projection_BrainstemCorticospinalTractR",
    "Projection_BrainstemMedialForebrainBundleL",
    "Projection_BrainstemMedialForebrainBundleR",
    "Projection_BrainstemMedialLemniscusL",
    "Projection_BrainstemMedialLemniscusR",
    "Projection_BrainstemNonDecussatingDentatorubrothalamicTractL",
    "Projection_BrainstemNonDecussatingDentatorubrothalamicTractR",
    "Projection_BrainstemReticularTractL",
    "Projection_BrainstemReticularTractR",
]

In [17]:
def get_bundle_from_features(col):
    rest = col[len("FEATURES_"):]
    parts = rest.split("_")
    bundle = "_".join(parts[:2])
    return bundle

In [18]:
# identify feature columns and map them to bundles
feature_cols = [c for c in df_final.columns if c.startswith("FEATURES_")]
keep_feature_cols = [c for c in feature_cols if get_bundle_from_features(c) in bundles_to_keep]
non_feature_cols = [c for c in df_final.columns if not c.startswith("FEATURES_")]
df_subset = df_final[non_feature_cols + keep_feature_cols].copy()

print(f"Total subjects before dropping NaNs: {df_subset.shape[0]}")
print(f"Total columns (including IDs, SES, demo, features): {df_subset.shape[1]}")

# subject-level missingness and complete cases
n_missing_per_subject = df_subset.isna().sum(axis=1)

print("\n=== Distribution of # of NaNs per subject (after bundle restriction) ===")
print(n_missing_per_subject.value_counts().sort_index())

n_subjects_total = df_subset.shape[0]
n_subjects_with_nan = (n_missing_per_subject > 0).sum()
n_subjects_complete = (n_missing_per_subject == 0).sum()

print(f"\nTotal subjects: {n_subjects_total}")
print(f"Subjects with ANY missing value: {n_subjects_with_nan}")
print(f"Subjects with NO missing values (complete cases): {n_subjects_complete}")

df_subset_complete = df_subset.dropna(axis=0, how="any").copy()

print(f"\nDropping subjects with any NaN after bundle restriction...")
print(f"Dropped {n_subjects_total - df_subset_complete.shape[0]} subjects")
print(f"Final sample size: {df_subset_complete.shape[0]} subjects")

Total subjects before dropping NaNs: 2980
Total columns (including IDs, SES, demo, features): 659

=== Distribution of # of NaNs per subject (after bundle restriction) ===
0      1643
2        30
3         7
6         5
12        2
18        1
39        1
45        1
51        1
54        1
57        1
72        2
93     1237
95       32
129       2
153       1
162       1
165       1
183       1
201       3
219       1
237       1
243       1
255       2
273       2
Name: count, dtype: int64

Total subjects: 2980
Subjects with ANY missing value: 1337
Subjects with NO missing values (complete cases): 1643

Dropping subjects with any NaN after bundle restriction...
Dropped 1337 subjects
Final sample size: 1643 subjects


In [20]:
def summarize_hbn_demographics(df, age_col="Age", sex_col="Sex", sample_label="HBN"):
    df = df.copy()

    df[age_col] = pd.to_numeric(df[age_col], errors="coerce")
    df[sex_col] = pd.to_numeric(df[sex_col], errors="coerce")

    n = df.shape[0]
    age_m = df[age_col].mean()
    age_s = df[age_col].std(ddof=1)

    n_f = int((df[sex_col] == 1).sum())
    n_m = int((df[sex_col] == 0).sum())
    n_known = n_f + n_m

    pct_f = 100 * n_f / n_known if n_known > 0 else np.nan
    pct_m = 100 * n_m / n_known if n_known > 0 else np.nan

    return pd.DataFrame([{
        "Sample": sample_label,
        "n": n,
        "Age, mean (SD)": f"{age_m:.2f} ({age_s:.2f})",
        "Female, n (%)": f"{n_f} ({pct_f:.1f}%)",
        "Male, n (%)": f"{n_m} ({pct_m:.1f}%)"
    }])

hbn_demo_table = summarize_hbn_demographics(df_subset_complete, age_col="Age", sex_col="Sex")
hbn_demo_table

,Sample,n,"Age, mean (SD)","Female, n (%)","Male, n (%)"
0,HBN,1643,10.40 (3.41),574 (34.9%),1069 (65.1%)


In [21]:
df_subset_complete["participant_id"].nunique()

1643

In [22]:
output_path = output_dir / "final_hbn_dataset_12_23.csv"
df_subset_complete.to_csv(output_path)